Быков Владимир Андреевич 465327 J3113

In [59]:
import numpy as np
import scipy.stats as stats
import plotly.graph_objects as go
import plotly.express as px
import seaborn as sns
import pandas as pd
from tqdm import tqdm

Ход выполнения работы
1. Сгенерировал выборку из нормального распределения (N=500)
2. Рассчитал базовые статистики и сравнение с теорией
3. Реализовал алгоритм бутстрапа (B=1000)
4. Построил доверительные интервалы для разных уровней значимости
5. Исследовал зависимость от объема выборки и числа итераций
6. Проверил покрытие доверительных интервалов

In [60]:
np.random.seed(42)
N = 500
mu_true = 2
sigma_true = 1.5
sample = np.random.normal(loc=mu_true, scale=sigma_true, size=N)

сгененрировали выборку размера N из нормального распредеелния 

In [61]:
# Расчет теоретических значений
theoretical = {
    "Mean": mu_true,
    "Median": mu_true,
    "Variance": sigma_true ** 2,
    "IQR": stats.norm.ppf(0.75, loc=mu_true, scale=sigma_true) 
            - stats.norm.ppf(0.25, loc=mu_true, scale=sigma_true)
}

# Расчет выборочных статистик
empirical = {
    "Mean": np.mean(sample),
    "Median": np.median(sample),
    "Variance": np.var(sample, ddof=1),
    "IQR": stats.iqr(sample)
}

# Вывод результатов
print("Теоретические значения:")
for k, v in theoretical.items():
    print(f"{k}: {v:.4f}")

print("\nЭмпирические оценки:")
for k, v in empirical.items():
    print(f"{k}: {v:.4f}")

Теоретические значения:
Mean: 2.0000
Median: 2.0000
Variance: 2.2500
IQR: 2.0235

Эмпирические оценки:
Mean: 2.0103
Median: 2.0192
Variance: 2.1664
IQR: 2.0056


вообще отклонения на 1\10 достаточно большое, но в контексте нашей выборки можно считать его достаточно точным

Посмотрим на выборк с разным количеством бинов (автоматически, при помощи Freedman-Diaconis rule и через рандомное число(у меня 7))

Правило Фридмана—Дьякониса ('fd')
Этот метод предлагает оптимальную ширину бина, которая зависит от:

Межквартильного размаха (IQR) – разницы между 75-м и 25-м процентилями.

Объёма данных (n) – количества наблюдений.

формула:
ширина бина = 2*(IQR)/(n^(1/3))

используем когда

Когда данные имеют нестандартное распределение (не нормальное).

Когда нужно автоматически выбрать разумное число бинов без ручного подбора.

Когда есть выбросы, так как IQR устойчив к ним.

In [62]:

bin_settings = ["auto", "fd", 7]
fig = go.Figure()

for bins in bin_settings:
    hist_data = np.histogram(sample, bins=bins, density=True)
    fig.add_trace(go.Bar(
        x=hist_data[1],
        y=hist_data[0],
        name=f"Бины: {bins}",
        opacity=0.6
    ))

x = np.linspace(mu_true - 4 * sigma_true, mu_true + 4 * sigma_true, 100)
fig.add_trace(go.Scatter(
    x=x,
    y=stats.norm.pdf(x, mu_true, sigma_true),
    mode='lines',
    name='Теоретическая плотность',
    line=dict(color='red')
))

fig.update_layout(
    title="Гистограммы с разными бинами",
    barmode='overlay',
    xaxis_title="Значение",
    yaxis_title="Плотность"
)
fig.show()

In [63]:
B = 1000
bootstrap_stats = {
    "Mean": [],
    "Median": [],
    "Variance": [],
    "IQR": []
}

for ш in tqdm(range(B)):
    bs_sample = np.random.choice(sample, size=N, replace=True)
    bootstrap_stats["Mean"].append(np.mean(bs_sample))
    bootstrap_stats["Median"].append(np.median(bs_sample))
    bootstrap_stats["Variance"].append(np.var(bs_sample, ddof=1))
    bootstrap_stats["IQR"].append(stats.iqr(bs_sample))

100%|██████████| 1000/1000 [00:00<00:00, 2526.09it/s]


In [64]:
fig = go.Figure()

for stat, values in bootstrap_stats.items():
    fig.add_trace(go.Histogram(
        x=values,
        name=f"Бутстрап {stat}",
        opacity=0.7,
        nbinsx=30
    ))
    fig.add_vline(
        x=empirical[stat],
        line_dash="dash",
        line_color="red",
        annotation_text=f"Эмпирическое {stat}",
        annotation_position="top right"
    )

fig.update_layout(
    title="Бутстрап-распределения статистик",
    xaxis_title="Значение",
    yaxis_title="Частота",
    barmode="overlay"
)
fig.show()

Точность оценки: Чем уже ДИ, тем точнее оценка.

Надёжность вывода: Если ДИ не включает нуль (для разницы средних), эффект статистически значим.


In [65]:
alphas = [0.1, 0.05, 0.01]
confidence_intervals = {}

for stat in bootstrap_stats:
    lower = np.percentile(bootstrap_stats[stat], [100 * a/2 for a in alphas])
    upper = np.percentile(bootstrap_stats[stat], [100 * (1 - a / 2) for a in alphas])
    confidence_intervals[stat] = list(zip(lower, upper))
    
# среденее и медиана
fig = go.Figure()

for i, stat in enumerate(["Mean", "Median"]):
    intervals = confidence_intervals[stat]
    for j, (a, (low, high)) in enumerate(zip(alphas, intervals)):
        fig.add_shape(
            type="line",
            x0=low, x1=high, y0=j, y1=j,
            line=dict(color="blue", width=2),
            xref="x", yref="y",
            name=f"{100 * (1 - a)}% ДИ"
        )
        fig.add_scatter(
            x=[empirical[stat]], y=[j],
            mode="markers",
            marker=dict(color="red", size=10),
            name=f"Эмпирическое {stat}"
        )

fig.update_layout(
    title="Доверительные интервалы для среднего и медианы",
    xaxis_title="Значение",
    yaxis_title="Уровень доверия",
    yaxis=dict(
        tickvals=list(range(len(alphas))),
        ticktext=[f"{100 * (1 - a)}%" for a in alphas]
    )
)
fig.show()

95% доверительный интервал (ДИ) — это диапазон значений, который с вероятностью 95% содержит истинный параметр генеральной совокупности (например, среднее значение, долю или разницу между группами).

Ширина 95% ДИ — это разница между верхней и нижней границами этого интервала. (upper-lower)

In [66]:
N_values = [50, 100, 200, 500, 1000]
B_fixed = 1000
ci_widths = []

for n in N_values:
    sample_n = np.random.normal(mu_true, sigma_true, n)
    bs_means = [np.mean(np.random.choice(sample_n, n, replace=True)) for _ in range(B_fixed)]
    lower, upper = np.percentile(bs_means, [2.5, 97.5])
    ci_widths.append(upper - lower)
    
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=N_values, y=ci_widths,
    mode="lines+markers",
    name="Ширина 95% ДИ"
))
fig.update_layout(
    title="Зависимость ширины доверительного интервала от N",
    xaxis_title="Объем выборки (N)",
    yaxis_title="Ширина 95% ДИ"
)
fig.show()

при увеличении N ДИ сужается.

In [67]:
# Heatmap покрытия (Plotly)
np.random.seed(42)
coverage_data = []
mu_target = 0
sigma_target = 1

for N in tqdm([50, 100, 200, 500, 1000]):
    for B in [100, 200, 400, 1600, 3200]:
        covered = 0
        for _ in range(100):
            sample = np.random.normal(mu_target, sigma_target, N)
            bs_means = [np.mean(np.random.choice(sample, N, replace=True)) for _ in range(B)]
            lower, upper = np.percentile(bs_means, [2.5, 97.5])
            if lower <= mu_target <= upper:
                covered += 1
        coverage_data.append([N, B, covered/100])

df = pd.DataFrame(coverage_data, columns=["N", "B", "Coverage"])
pivot_table = df.pivot(index="N", columns="B", values="Coverage")

fig = px.imshow(
    pivot_table,
    labels=dict(x="B (число бутстрап-выборок)", y="N (размер выборки)", color="Доля покрытия"),
    color_continuous_scale=px.colors.sequential.Viridis,  # Используем встроенную шкалу Plotly
    zmin=0.8,
    zmax=1.0,
    text_auto=True
)
fig.update_layout(title="Доля покрытия истинного среднего")
fig.show()

100%|██████████| 5/5 [01:14<00:00, 14.96s/it]


Заключение
1. Бутстрап-оценки: Позволяют оценить распределение статистик даже для малых выборок
2. Доверительные интервалы: Ширина уменьшается с ростом N и стабилизируется при B > 1000
3. Покрытие интервалов: Фактическая доля покрытия близка к номинальной (95%) при N ≥ 200
4. Сравнение статистик: Медиана имеет более широкие ДИ по сравнению со средним
5. Практические рекомендации: Для надежных оценок использовать B ≥ 1000 и N ≥ 200